# 랭체인 멀티턴

- LangChain의 RunnableWithMessageHistory를 사용하여 대화기록을 관리하는 챗봇 구현
- 세션 ID별로 대화 기록이 어떻게 분리되어 저장되는지 확인

## 라이브러리 불러오기

In [1]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼wrapper 클래스
from langchain_openai import ChatOpenAI  # 오픈AI 모델을 사용하는 랭체인 챗봇 클래스
from langchain_core.messages import HumanMessage

## API KEY 불러오기

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()

True

## 대화 기록 함수 정의

In [4]:
model = ChatOpenAI(model="gpt-4o-mini")

# 세션별 대화 기록을 저장할 딕셔너리
store = {}

# 세션 ID에 따라 대화 기록을 가져오는 함수
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

## 첫 번째 대화

In [29]:
config = {"configurable": {"session_id": "test_new_001"}}

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 내 이름은 최은혜야.")],
    config=config
)

print(response.content)

Parent run 32699c1d-c9f4-440a-bdb5-6d6ce2d8011b not found for run 805a9cc8-93e2-4043-863f-da2ed7e16180. Treating as a root run.


안녕, 최은혜님! 만나서 반가워요. 어떻게 도와드릴까요?


## 두 번째 대화

In [31]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

Parent run ac764dc8-c3a9-4a0e-962e-279c8e7dc4bd not found for run 1591f018-38fd-4306-bfa6-024a73bd33a0. Treating as a root run.


당신의 이름은 최은혜입니다! 다른 질문이 있거나 이야기하고 싶은 것이 있다면 말씀해 주세요.


## 세션 테스트

In [35]:
config = {"configurable": {"session_id": "test_new_002"}}

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

response.content

Parent run d36f46aa-6a07-44a7-88be-1b828be82f80 not found for run 0199b203-1a80-4ff4-b0f5-0858fd0c7000. Treating as a root run.


'죄송하지만, 당신의 이름은 알 수 없습니다. 당신이 원하신다면 이름을 알려주실 수 있습니다!'

In [36]:
config = {"configurable": {"session_id": "test_new_001"}}

response = with_message_history.invoke(
    [HumanMessage(content="아까 우리가 무슨 얘기 했지?")],
    config=config,
)

response.content

Parent run 6b241a85-563d-428d-947c-682db88b2f2a not found for run 0ef85067-8bf1-4781-a2ab-a9c8885ae27d. Treating as a root run.


'우리는 당신의 이름인 최은혜에 대해 이야기했습니다. 더 이야기하고 싶은 주제가 있나요?'

## 스트리밍 응답 출력

In [38]:
config = {"configurable": {"session_id": "test_new_001"}}
for r in with_message_history.stream(
    [HumanMessage(content = "내가 어느 나라 사람인지 맞춰보고, 그 나라의 문화에 대해 말해봐")],
    config=config,
):
    print(r.content, end="")
    # print(r.content, end="|")

Parent run 547e975e-e84e-49fe-b814-64a30879f0e5 not found for run 7181bc8b-5bd3-45b5-99b4-a2673c679704. Treating as a root run.


최은혜라는 이름은 한국에서 흔히 사용되는 이름입니다. 만약 한국이 맞다면, 한국의 문화에 대해 몇 가지 말씀드릴게요.

1. **음식**: 한국 음식은 다양하고 맛있습니다. 김치, 비빔밥, 불고기와 같은 전통 음식이 유명하며, 최근에는 K-푸드가 전 세계적으로 인기를 얻고 있습니다.

2. **전통 의상**: 한복은 한국의 전통 의상으로, 특별한 행사나 명절에 입습니다. 화려한 색상과 독특한 디자인이 특징입니다.

3. **K-POP과 K-드라마**: 한국의 대중문화는 K-POP과 K-드라마로 많이 알려져 있습니다. BTS, BLACKPINK와 같은 아이돌 그룹과 인기 있는 드라마들이 세계적으로 큰 인기를 얻고 있습니다.

4. **명절**: 설날과 추석은 한국의 중요한 명절로, 가족들이 모여 함께 음식을 나누고 조상을 기리는 특별한 시간입니다.

혹시 한국이 맞나요? 다른 나라에 대해 이야기하고 싶다면 말씀해 주세요!